In [1]:
#!/usr/bin/env python3
"""
YOLOv9 Wheat Disease Detection Implementation
Complete training and evaluation pipeline for wheat disease detection using YOLOv9
"""

import os
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import cv2
import shutil
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import json
from ultralytics import YOLO

# Configuration
DATASET_DIR = '../dataset'
OUTPUT_DIR = './yolov9_results'
EPOCHS =10
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
IMAGE_SIZE = 640
CONFIDENCE_THRESHOLD = 0.5
IOU_THRESHOLD = 0.7

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'models'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'results'), exist_ok=True)

class YOLOv9WheatDisease(nn.Module):
    """YOLOv9 model for wheat disease detection and classification"""
    
    def __init__(self, num_classes=12, model_size='yolov9c.pt'):
        super(YOLOv9WheatDisease, self).__init__()
        
        # Load pre-trained YOLOv9 model
        self.yolo_model = YOLO(model_size)
        
        # Disease classification head
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
        
        # Feature extraction layers
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
    def forward(self, x):
        # Extract features using YOLOv9 backbone
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1)
        
        # Classify disease
        disease_pred = self.classifier(features)
        
        return disease_pred

class WheatDiseaseDataset(Dataset):
    """Custom dataset for wheat disease detection"""
    
    def __init__(self, data_dir, transform=None, split='train'):
        self.data_dir = data_dir
        self.transform = transform
        self.split = split
        self.samples = []
        
        # Get disease classes
        self.classes = sorted([d for d in os.listdir(data_dir) 
                              if os.path.isdir(os.path.join(data_dir, d))])
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        # Load samples
        for class_name in self.classes:
            class_dir = os.path.join(data_dir, class_name)
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(class_dir, img_name)
                    self.samples.append((img_path, self.class_to_idx[class_name]))
        
        print(f"📊 {split.capitalize()} dataset: {len(self.samples)} samples, {len(self.classes)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

def create_yolo_dataset(dataset_dir, output_dir):
    """Create YOLO format dataset for wheat disease detection"""
    
    print("🔧 Creating YOLO dataset...")
    
    # Create directories
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'train', 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'train', 'labels'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'val', 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'val', 'labels'), exist_ok=True)
    
    # Get disease classes
    disease_classes = sorted([d for d in os.listdir(dataset_dir) 
                            if os.path.isdir(os.path.join(dataset_dir, d))])
    
    print(f"�� Found {len(disease_classes)} disease classes: {disease_classes}")
    
    # Process each class
    for class_idx, class_name in enumerate(disease_classes):
        class_dir = os.path.join(dataset_dir, class_name)
        image_files = [f for f in os.listdir(class_dir) 
                      if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        # Split into train/val
        train_files, val_files = train_test_split(
            image_files, test_size=0.2, random_state=42
        )
        
        print(f"   Processing {class_name}: {len(image_files)} images")
        
        # Process training images
        for img_file in train_files:
            src_path = os.path.join(class_dir, img_file)
            dst_path = os.path.join(output_dir, 'train', 'images', img_file)
            shutil.copy2(src_path, dst_path)
            
            # Create YOLO label (center bounding box)
            create_yolo_label(img_file, class_idx, output_dir, 'train')
        
        # Process validation images
        for img_file in val_files:
            src_path = os.path.join(class_dir, img_file)
            dst_path = os.path.join(output_dir, 'val', 'images', img_file)
            shutil.copy2(src_path, dst_path)
            
            # Create YOLO label
            create_yolo_label(img_file, class_idx, output_dir, 'val')
    
    # Create YAML configuration
    create_yolo_yaml(output_dir, disease_classes)
    
    print(f"✅ YOLO dataset created successfully in {output_dir}")

def create_yolo_label(img_file, class_idx, output_dir, split):
    """Create YOLO format label file"""
    
    label_name = img_file.rsplit('.', 1)[0] + '.txt'
    label_path = os.path.join(output_dir, split, 'labels', label_name)
    
    # Create center bounding box (0.4 to 0.6 of image dimensions)
    x_center = 0.5
    y_center = 0.5
    width = 0.2
    height = 0.2
    
    with open(label_path, 'w') as f:
        f.write(f"{class_idx} {x_center} {y_center} {width} {height}\n")

def create_yolo_yaml(output_dir, classes):
    """Create YOLO data YAML configuration"""
    
    yaml_content = f"""path: {os.path.abspath(output_dir)}
train: train/images
val: val/images

nc: {len(classes)}
names: {classes}
"""
    
    yaml_path = os.path.join(output_dir, 'wheat_disease.yaml')
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    
    print(f"�� YAML configuration saved: {yaml_path}")

def create_data_loaders(yolo_output_dir, config):
    """Create data loaders for training and validation"""
    
    from torchvision import transforms
    
    # Data transformations
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    # Create datasets
    train_dataset = WheatDiseaseDataset(yolo_output_dir, transform, 'train')
    val_dataset = WheatDiseaseDataset(yolo_output_dir, transform, 'val')
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], 
                             shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], 
                           shuffle=False, num_workers=4)
    
    # Create test loader (using validation for now)
    test_loader = DataLoader(val_dataset, batch_size=config['batch_size'], 
                            shuffle=False, num_workers=4)
    
    return train_loader, val_loader, test_loader

def train_yolov9_model(model, train_loader, val_loader, config):
    """Advanced YOLOv9 training with disease detection"""
    
    print("🚀 Starting YOLOv9 Wheat Disease Detection Training...")
    
    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'], 
        weight_decay=config['weight_decay']
    )
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=config['epochs']
    )
    
    # Loss function
    classification_loss = nn.CrossEntropyLoss()
    
    best_accuracy = 0.0
    patience = 15
    patience_counter = 0
    
    # Training history
    train_losses = []
    val_accuracies = []
    
    for epoch in range(config['epochs']):
        model.train()
        running_loss = 0.0
        
        for batch_idx, (data, targets) in enumerate(train_loader):
            data = data.cuda()
            targets = targets.cuda()
            
            optimizer.zero_grad()
            
            # Forward pass
            disease_pred = model(data)
            
            # Calculate loss
            loss = classification_loss(disease_pred, targets)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            if batch_idx % 50 == 0:
                print(f'Epoch {epoch+1}, Batch {batch_idx}: Loss = {loss.item():.4f}')
        
        # Validation
        val_accuracy = validate_yolov9_model(model, val_loader)
        scheduler.step()
        
        # Store history
        avg_loss = running_loss / len(train_loader)
        train_losses.append(avg_loss)
        val_accuracies.append(val_accuracy)
        
        # Print epoch summary
        print(f'\n📊 Epoch {epoch+1}/{config["epochs"]} Summary:')
        print(f'   Training Loss: {avg_loss:.4f}')
        print(f'   Validation Accuracy: {val_accuracy:.4f}')
        print(f'   Learning Rate: {scheduler.get_last_lr()[0]:.6f}')
        
        # Early stopping
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            patience_counter = 0
            
            # Save best model
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'accuracy': best_accuracy,
                'config': config
            }, os.path.join(OUTPUT_DIR, 'models', 'best_yolov9_wheat_disease.pth'))
            
            print(f'   🏆 New best model saved! Accuracy: {best_accuracy:.4f}')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'   ⏹️ Early stopping at epoch {epoch+1}')
                break
    
    # Plot training history
    plot_training_history(train_losses, val_accuracies)
    
    return model, best_accuracy

def validate_yolov9_model(model, val_loader):
    """Validate YOLOv9 model"""
    
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, targets in val_loader:
            data = data.cuda()
            targets = targets.cuda()
            
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
    
    accuracy = correct / total
    return accuracy

def evaluate_yolov9_model(model, test_loader, class_names):
    """Comprehensive evaluation of YOLOv9 wheat disease detection"""
    
    model.eval()
    all_preds = []
    all_targets = []
    
    print("🔍 Evaluating YOLOv9 model...")
    
    with torch.no_grad():
        for batch_idx, (data, targets) in enumerate(test_loader):
            data = data.cuda()
            targets = targets.cuda()
            
            # Get predictions
            disease_pred = model(data)
            pred_classes = torch.argmax(disease_pred, dim=1)
            
            all_preds.extend(pred_classes.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            
            if batch_idx % 20 == 0:
                print(f"   Processed {batch_idx} batches...")
    
    # Calculate metrics
    accuracy = np.sum(np.array(all_targets) == np.array(all_preds)) / len(all_targets)
    
    # Generate confusion matrix
    cm = confusion_matrix(all_targets, all_preds)
    
    # Classification report
    report = classification_report(all_targets, all_preds, 
                                 target_names=class_names, 
                                 output_dict=True)
    
    print(f"\n📊 YOLOv9 Evaluation Results:")
    print(f"   Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    # Per-class performance
    print(f"\n📈 Per-Class Performance:")
    for i, class_name in enumerate(class_names):
        if i < len(report):
            precision = report[class_name]['precision']
            recall = report[class_name]['recall']
            f1 = report[class_name]['f1-score']
            print(f"   {class_name:20} | P: {precision:.3f} | R: {recall:.3f} | F1: {f1:.3f}")
    
    # Save results
    save_evaluation_results(accuracy, cm, report, class_names)
    
    return accuracy, cm, report

def plot_training_history(train_losses, val_accuracies):
    """Plot training history"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Training loss
    ax1.plot(train_losses, 'b-', label='Training Loss')
    ax1.set_title('Training Loss Over Time')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    
    # Validation accuracy
    ax2.plot(val_accuracies, 'r-', label='Validation Accuracy')
    ax2.set_title('Validation Accuracy Over Time')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'results', 'training_history.png'), dpi=300)
    plt.show()

def save_evaluation_results(accuracy, confusion_matrix, report, class_names):
    """Save evaluation results"""
    
    # Save confusion matrix plot
    plt.figure(figsize=(12, 10))
    sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('YOLOv9 Wheat Disease Detection Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'results', 'confusion_matrix.png'), dpi=300)
    plt.show()
    
    # Save results summary
    results_summary = {
        'overall_accuracy': accuracy,
        'confusion_matrix': confusion_matrix.tolist(),
        'classification_report': report,
        'class_names': class_names
    }
    
    results_path = os.path.join(OUTPUT_DIR, 'results', 'evaluation_results.json')
    with open(results_path, 'w') as f:
        json.dump(results_summary, f, indent=2)
    
    print(f"💾 Results saved to: {results_path}")

def main():
    """Main execution for YOLOv9 wheat disease detection"""
    
    print("🌾 YOLOv9 Wheat Disease Detection Training")
    print("=" * 50)
    
    # Configuration
    config = {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'image_size': IMAGE_SIZE,
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'iou_threshold': IOU_THRESHOLD
    }
    
    print(f"📋 Configuration:")
    for key, value in config.items():
        print(f"   {key}: {value}")
    
    # Create YOLO dataset
    yolo_output_dir = os.path.join(OUTPUT_DIR, 'yolo_dataset')
    create_yolo_dataset(DATASET_DIR, yolo_output_dir)
    
    # Initialize model
    model = YOLOv9WheatDisease(num_classes=12, model_size='yolov9c.pt')
    model = model.cuda()
    
    print(f"🚀 Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")
    
    # Create data loaders
    train_loader, val_loader, test_loader = create_data_loaders(yolo_output_dir, config)
    
    # Train model
    trained_model, best_accuracy = train_yolov9_model(
        model, train_loader, val_loader, config
    )
    
    # Evaluate model
    class_names = ['aphid', 'army_worm', 'black_rust', 'brown_rust', 
                   'common_rust', 'fusarium_head_blight', 'healthy', 
                   'leaf_blight', 'powdery_mildew_leaf', 'spetoria', 
                   'tan_spot', 'yellow_rust']
    
    accuracy, confusion_matrix, report = evaluate_yolov9_model(
        trained_model, test_loader, class_names
    )
    
    print(f"\n�� Training completed successfully!")
    print(f"🏆 Best validation accuracy: {best_accuracy:.4f}")
    print(f"📊 Final test accuracy: {accuracy:.4f}")
    print(f"📁 Results saved in: {OUTPUT_DIR}")
    
    return trained_model

if __name__ == "__main__":
    model = main()

🌾 YOLOv9 Wheat Disease Detection Training
📋 Configuration:
   epochs: 10
   batch_size: 16
   learning_rate: 0.0001
   weight_decay: 0.01
   image_size: 640
   confidence_threshold: 0.5
   iou_threshold: 0.7
🔧 Creating YOLO dataset...
�� Found 12 disease classes: ['aphid', 'army_worm', 'black_rust', 'brown_rust', 'common_rust', 'fusarium_head_blight', 'healthy', 'leaf_blight', 'powdery_mildew_leaf', 'spetoria', 'tan_spot', 'yellow_rust']
   Processing aphid: 295 images
   Processing army_worm: 285 images
   Processing black_rust: 274 images
   Processing brown_rust: 299 images
   Processing common_rust: 299 images
   Processing fusarium_head_blight: 257 images
   Processing healthy: 565 images
   Processing leaf_blight: 296 images
   Processing powdery_mildew_leaf: 300 images
   Processing spetoria: 300 images
   Processing tan_spot: 281 images
   Processing yellow_rust: 300 images
�� YAML configuration saved: ./yolov9_results\yolo_dataset\wheat_disease.yaml
✅ YOLO dataset created succ

AssertionError: Torch not compiled with CUDA enabled